# Speech Recognition (ASR)

**Module:** 16 — Speech AI

Pipelines, metrics, streaming, and domain adaptation for speech-to-text.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain ASR pipeline concepts and modern E2E models
- Compute and interpret WER-related metrics
- Design streaming ASR with partial hypotheses
- Apply domain adaptation tactics (boosting, finetune, biasing)


## ASR Pipeline Concepts

### Definition
ASR maps audio to text (and often word timestamps / confidence).

### Why it matters
Transcripts unlock search, agents, compliance, captions — garbage ASR poisons everything downstream.

### How it works
Classic: acoustic model + lexicon + LM. Modern E2E: encoder-decoder / CTC / RNN-T / Whisper-like seq2seq on mel features.

### Intuition
Listening exams with a spellchecker — acoustic evidence + language prior.

### Pitfalls
- Evaluating only on clean read speech
- No punctuation/diarization when product needs both

### When to use
Captions, voice bots, meeting notes, analytics.


### Metrics & features

| Metric | Meaning | Watch-out |
|--------|---------|-----------|
| WER | Word errors / ref words | Unfair on short utts |
| CER | Char errors | Useful for names/codes |
| RTFx | Audio_sec / process_sec | Throughput, not latency |
| Partial lag | Time to useful partial | Streaming UX |

```mermaid
flowchart LR
  A[Audio] --> F[Features]
  F --> E[Encoder]
  E --> D[Decoder / LM]
  D --> T[Text + times]
```


In [ ]:
# Demo 1: WER via word-level Levenshtein
def wer(ref: str, hyp: str) -> float:
    r, h = ref.lower().split(), hyp.lower().split()
    if not r: return 0.0 if not h else 1.0
    dp = list(range(len(h)+1))
    for i in range(1, len(r)+1):
        prev, dp[0] = dp[0], i
        for j in range(1, len(h)+1):
            cur = dp[j]
            dp[j] = prev if r[i-1]==h[j-1] else 1+min(prev, dp[j], dp[j-1])
            prev = cur
    return dp[-1] / len(r)

print(wer("the flight to paris is on time", "the flight to paris is on time"))
print(round(wer("the flight to paris is on time", "flight to paris on time"), 3))
print(round(wer("cancel my order", "cancel my odor"), 3))


In [ ]:
# Demo 2: OpenAI transcription request shape
import os, json
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
req = {
    "model": "whisper-1",
    "file": "@call.wav",
    "response_format": "verbose_json",
    "timestamp_granularities": ["word"],
    "language": "en",
}
mock_resp = {
    "text": "Cancel my order number 4455.",
    "language": "en",
    "duration": 2.4,
    "words": [
        {"word": "Cancel", "start": 0.0, "end": 0.35},
        {"word": "my", "start": 0.36, "end": 0.45},
        {"word": "order", "start": 0.46, "end": 0.8},
    ],
}
print(json.dumps(req, indent=2))
print(json.dumps(mock_resp, indent=2))
print("key", OPENAI_API_KEY[:12] + "...")


## Streaming ASR

### Definition
Streaming ASR emits partial hypotheses before the user finishes speaking.

### Why it matters
Duplex agents and live captions cannot wait for end-of-utterance finals only.

### How it works
Use streaming APIs / RNN-T style models; stabilize partials; endpoint with VAD + semantic completeness.

### Intuition
Subtitles that update as someone talks — expect flicker; manage it.

### Pitfalls
- Showing every unstable partial in UI
- Finalizing too early on pauses

### When to use
Voice agents, live captions, real-time translation.


In [ ]:
# Demo 3: partial hypothesis stabilizer
class PartialStabilizer:
    def __init__(self, hold=2):
        self.hold = hold
        self.last = ""
        self.stable_count = 0
        self.committed = ""
    def update(self, partial: str) -> str:
        if partial.startswith(self.last) or self.last.startswith(partial):
            self.stable_count += 1
        else:
            self.stable_count = 0
        self.last = partial
        if self.stable_count >= self.hold:
            self.committed = partial
        return self.committed

st = PartialStabilizer()
for p in ["can", "cancel", "cancel my", "cancel my ord", "cancel my order"]:
    print(p, "=>", st.update(p))


In [ ]:
# Demo 4: endpointing heuristic
def should_finalize(silence_ms, partial, min_silence=400, min_words=2):
    if silence_ms < min_silence: return False
    if len(partial.split()) < min_words: return False
    if partial.strip().endswith((",", "and", "or", "to")): return False
    return True
print(should_finalize(500, "cancel my order"))
print(should_finalize(500, "I want to"))
print(should_finalize(200, "cancel my order"))


## Domain Adaptation

### Definition
Adaptation specializes ASR to your vocabulary, accents, channel, and noise.

### Why it matters
Generic models butcher SKUs, drug names, and brand terms.

### How it works
Tactics: phrase boosting/biasing, custom LM hotwords, finetuning, audio enhancement, locale routing.

### Intuition
Teach the model your company's weird words.

### Pitfalls
- Boosting everything (noise)
- Finetune without held-out call audio

### When to use
Vertical voice bots and specialized transcription.


In [ ]:
# Demo 5: hotword boost scorer (toy)
def rescore(hypotheses, hotwords, boost=0.5):
    scored = []
    for h, base in hypotheses:
        bonus = sum(boost for w in hotwords if w.lower() in h.lower())
        scored.append((base + bonus, h))
    return sorted(scored, reverse=True)

hyps = [("cancel my odor 4455", 0.6), ("cancel my order 4455", 0.58), ("cancel my offer 4455", 0.55)]
print(rescore(hyps, ["order", "4455"]))


In [ ]:
# Demo 6: normalize transcripts for fair WER
import re
def normalize(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s
print(normalize("Cancel my order, #4455!"))
print(wer(normalize("Cancel my order 4455"), normalize("cancel my order, #4455!")))


### Feature checklist for production ASR

| Feature | Why |
|---------|-----|
| Word timestamps | Align UI / redact spans |
| Confidence | Route low-conf to HITL |
| Language detect | Multilingual lines |
| Punctuation | LLM readability |
| Speaker labels | Optional; see diarization |


### Checklist — ASR launch

- [ ] Eval set from real channel audio
- [ ] Hotwords list owned by product
- [ ] Streaming partial UX reviewed
- [ ] PII redaction on transcripts
- [ ] WER sliced by accent/noise


### Try it yourself — ASR metrics

1. Implement substitution/deletion/insertion counts.
2. Add a numeric-code CER metric for order IDs.
3. Simulate partial stream from a final string.

**Stretch:** Call a transcription API with YOUR_OPENAI_API_KEY if available.


### Try it yourself — Adaptation

1. Build a hotword list for a pharmacy bot.
2. Propose finetune data sampling strategy.


## Knowledge Check

**Q1.** Why normalize before WER?

<details><summary>Answer</summary>

Punctuation/case differences shouldn't dominate error rate for many products.

</details>

**Q2.** What is endpointing?

<details><summary>Answer</summary>

Deciding the user finished speaking so the system can act — often VAD + linguistic cues.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `WER` | Word error rate |
| `CTC` | Connectionist temporal classification |
| `RNN-T` | RNN Transducer streaming ASR family |
| `hotword` | Domain term boosted at decode |
| `endpointing` | Detect end of user utterance |
| `partial` | Interim ASR hypothesis |


## Key Takeaways

- ASR quality sets the ceiling for voice products
- Streaming needs stabilizer + careful endpointing
- Domain adaptation is mandatory for jargon
- Slice WER by channel, accent, and noise


## Production Incident Patterns — ASR

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "ASR",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — ASR

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("ASR", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — ASR ops

1. Draft an on-call runbook bullet list for ASR when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
